# DealBreaker — M&A Due Diligence AI Agent

Multi-agent due diligence pipeline: **Google ADK 2.3 · Gemini 2.0 Flash · Python 3.10+**

This notebook:
1. Clones the DealBreaker repo from GitHub and installs all dependencies
2. Loads API keys from Kaggle Secrets
3. Runs a full due diligence pipeline on **Salesforce** (a $50 M acquisition target)
4. Displays the risk-matrix chart and PDF download link inline

**Estimated runtime:** 5–10 minutes on Kaggle free CPU (7 agents, ~20 external API calls).

---

```
dealbreaker_coordinator (root LlmAgent)
└── due_diligence_pipeline (SequentialAgent)
    ├── investigation_phase (ParallelAgent)
    │   ├── financial_analyst      SEC EDGAR 10-K / DCF / Beneish M-Score
    │   ├── legal_reviewer         CourtListener + EDGAR + OFAC + PatentsView
    │   ├── market_researcher      EDGAR TAM + peer benchmarks
    │   ├── news_sentiment_analyst GDELT + Reddit
    │   └── people_culture_analyst EDGAR DEF 14A + Reddit
    └── synthesis_phase (SequentialAgent)
        ├── risk_assessor          5×5 risk matrix + dealbreaker flag
        └── report_generator       Executive summary + PDF
```

All data sources are **public** — no paywalled content is accessed.

## Before running — add Kaggle Secrets

Open **Kaggle → Add-ons → Secrets** and add the following keys:

| Secret name | Where to get it |
|-------------|----------------|
| `GOOGLE_API_KEY` | [aistudio.google.com](https://aistudio.google.com) — "Get API key" |
| `COURTLISTENER_API_KEY` | [courtlistener.com/sign-in](https://www.courtlistener.com/sign-in/) — free account |
| `CONTACT_EMAIL` | Your email address (required by SEC EDGAR Fair Access policy) |

> **Quota note:** The Gemini free tier allows ~15 req/min. A 7-agent parallel run can
> hit this limit and return `429 RESOURCE_EXHAUSTED`. Enable billing on your Google Cloud
> project for uninterrupted runs, or wait 24 h for the daily quota to reset.
>
> `COURTLISTENER_API_KEY` is optional — the pipeline runs without it; federal case
> search returns a graceful error and all other legal tools continue normally.

Also **update `REPO_URL`** in Step 1 to point to your fork of the repository.

In [ ]:
# ── Step 1: Clone repo and install all dependencies ───────────────────────────
# ⚠  Update REPO_URL to your fork before running.

REPO_URL = "https://github.com/YOUR_USERNAME/dealbreaker-ai-agent.git"

import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path(os.getcwd()) / "dealbreaker-ai-agent"


def _run(cmd, *, cwd=None, check=True, tail=8):
    """Run a shell command; print the last `tail` lines of stdout."""
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, cwd=cwd)
    out = (r.stdout + r.stderr).strip()
    if out:
        lines = out.splitlines()
        print("\n".join(lines[-tail:]))
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed (exit {r.returncode}): {cmd}")
    return r.returncode


# 1a — Clone (skip if already present)
if not REPO_DIR.exists():
    print("Cloning repository...")
    _run(f"git clone --depth 1 {REPO_URL} dealbreaker-ai-agent")
    print(f"✓ Cloned → {REPO_DIR}")
else:
    print(f"✓ Repo already present at {REPO_DIR} — skipping clone.")

# 1b — Main package (installs google-adk, httpx, structlog, presidio, …)
print("\nInstalling main package (≈ 2 min)...")
_run(f"{sys.executable} -m pip install -e . -q --no-warn-script-location",
     cwd=REPO_DIR)
print("✓ Main package installed")

# 1c — MCP server deps
print("\nInstalling MCP server dependencies...")
_run(
    f"{sys.executable} -m pip install"
    " -r mcp_servers/sec_edgar_server/requirements.txt -q",
    cwd=REPO_DIR,
)
_run(
    f"{sys.executable} -m pip install"
    " -r mcp_servers/legal_db_server/requirements.txt -q",
    cwd=REPO_DIR,
)
print("✓ MCP deps installed")

# 1d — spaCy model (≈ 580 MB, needed by PIIRedactionPlugin)
print("\nDownloading spaCy model en_core_web_lg (≈ 580 MB)...")
_run(f"{sys.executable} -m spacy download en_core_web_lg -q", cwd=REPO_DIR)
print("✓ spaCy model ready")

print("\n✅ All dependencies installed")

In [ ]:
# ── Step 2: Load API keys ─────────────────────────────────────────────────────

import os

_IS_KAGGLE = "KAGGLE_URL_BASE" in os.environ

if _IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient  # type: ignore[import]
    _sc = UserSecretsClient()

    def _secret(name, default=""):
        try:
            return _sc.get_secret(name)
        except Exception:
            print(f"  ⚠  Secret '{name}' not found — using default.")
            return default

    GOOGLE_API_KEY        = _secret("GOOGLE_API_KEY")
    COURTLISTENER_API_KEY = _secret("COURTLISTENER_API_KEY")
    CONTACT_EMAIL         = _secret("CONTACT_EMAIL", "kaggle-demo@example.com")
    print("✓ Secrets loaded from Kaggle Secrets")

else:
    # Local / Colab — set environment variables before running this cell, or
    # assign the strings directly here (never commit real keys to version control).
    GOOGLE_API_KEY        = os.getenv("GOOGLE_API_KEY", "")
    COURTLISTENER_API_KEY = os.getenv("COURTLISTENER_API_KEY", "")
    CONTACT_EMAIL         = os.getenv("CONTACT_EMAIL", "local-demo@example.com")
    print("✓ Keys read from environment variables")

if not GOOGLE_API_KEY:
    raise ValueError(
        "GOOGLE_API_KEY is empty.  "
        "Add it in Kaggle → Add-ons → Secrets with key name 'GOOGLE_API_KEY'."
    )

# Propagate into the environment so the ADK subprocess inherits them
os.environ["GOOGLE_API_KEY"]            = GOOGLE_API_KEY
os.environ["COURTLISTENER_API_KEY"]     = COURTLISTENER_API_KEY
os.environ["CONTACT_EMAIL"]             = CONTACT_EMAIL
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

_masked = f"{'*' * 8}{GOOGLE_API_KEY[-4:]}" if len(GOOGLE_API_KEY) > 4 else "****"
print(f"  GOOGLE_API_KEY       : {_masked}")
print(f"  COURTLISTENER_API_KEY: {'set' if COURTLISTENER_API_KEY else 'NOT SET (federal case search will be skipped)'}")
print(f"  CONTACT_EMAIL        : {CONTACT_EMAIL}")

In [ ]:
# ── Step 3: Write .env + replay.json ─────────────────────────────────────────
# Edit DEAL below to analyse a different company (any US-listed SEC filer).

import json
from pathlib import Path

REPO = Path(REPO_DIR)  # set in Step 1

# Write app/.env — ADK reads this file when loading the app
(REPO / "app" / ".env").write_text(
    f"GOOGLE_API_KEY={GOOGLE_API_KEY}\n"
    f"GOOGLE_GENAI_USE_VERTEXAI=False\n"
    f"COURTLISTENER_API_KEY={COURTLISTENER_API_KEY}\n"
    f"CONTACT_EMAIL={CONTACT_EMAIL}\n"
)
print("✓ app/.env written")

# ── Deal parameters ────────────────────────────────────────────────────────────
# Change 'target_company' and the other fields to run a different deal.
DEAL = {
    "target_company"      : "Salesforce",          # SEC-registered name
    "industry"            : "SaaS",
    "deal_value"          : 50_000_000,             # USD
    "deal_type"           : "acquisition",
    "deal_id"             : "kaggle_demo_001",      # determines output sub-directory
    "deal_jurisdictions"  : ["US"],
    "scope"               : (
        "Full acquisition due diligence — financial, legal, market, "
        "news/sentiment, and people/culture workstreams"
    ),
    "acquirer_description": (
        "Strategic acquirer in the enterprise software industry "
        "seeking to expand SaaS capabilities"
    ),
}

# replay.json pre-seeds ADK session state and provides the opening query
replay = {
    "state": DEAL,
    "queries": [
        f"Analyze {DEAL['target_company']} as an acquisition target. "
        f"Industry is {DEAL['industry']}. "
        f"Deal value is {DEAL['deal_value']:,} USD. "
        f"Deal ID is {DEAL['deal_id']}. "
        f"Jurisdictions are {', '.join(DEAL['deal_jurisdictions'])}."
    ],
}
(REPO / "replay.json").write_text(json.dumps(replay, indent=2))
print("✓ replay.json written")

print(f"\nDeal parameters:")
print(f"  Target : {DEAL['target_company']}")
print(f"  Type   : {DEAL['deal_type'].title()}")
print(f"  Value  : ${DEAL['deal_value']:,}")
print(f"  ID     : {DEAL['deal_id']}")
print(f"  Query  : {replay['queries'][0][:80]}...")

In [ ]:
# ── Step 4: Verify setup ──────────────────────────────────────────────────────

import shutil
import subprocess
import sys
from pathlib import Path

REPO = Path(REPO_DIR)

# 4a — syntax-check every module in app/
r = subprocess.run(
    [sys.executable, "-m", "compileall", "app/", "-q"],
    cwd=REPO, capture_output=True, text=True,
)
if r.returncode == 0:
    print("✓ py_compile: all app/ modules pass")
else:
    print("✗ Syntax errors detected:")
    print(r.stderr or r.stdout)

# 4b — locate the adk CLI
_adk = shutil.which("adk")
if _adk:
    print(f"✓ adk CLI : {_adk}")
else:
    # adk might be installed but not on PATH — try via the current interpreter
    probe = subprocess.run(
        [sys.executable, "-c", "from google.adk.cli.cli import cli; print('ok')"],
        cwd=REPO, capture_output=True, text=True,
    )
    if probe.returncode == 0:
        print("✓ adk importable via google.adk (will use 'python -m google.adk' fallback)")
        _adk = None  # flag: use python -m fallback in Step 5
    else:
        print("⚠  adk not found — check that 'pip install -e .' completed successfully")
        _adk = None

# 4c — check google-adk version
ver = subprocess.run(
    [sys.executable, "-c",
     "import importlib.metadata; print(importlib.metadata.version('google-adk'))"],
    cwd=REPO, capture_output=True, text=True,
)
print(f"✓ google-adk version: {ver.stdout.strip() or 'unknown'}")

# 4d — confirm reports/ will be created
expected_chart = REPO / "reports" / DEAL["deal_id"] / "risk_matrix.png"
print(f"\nExpected outputs after run:")
print(f"  Chart : {expected_chart}")
print(f"  PDF   : {expected_chart.parent / 'due_diligence_report.pdf'}")

print("\n✅ Ready — proceed to Step 5")

In [ ]:
# ── Step 5: Run the full pipeline ─────────────────────────────────────────────
# Output streams to this cell in real time.
# Hard cap: 12 minutes (Kaggle sessions allow up to 9 h, but 5–10 min is typical).

import os
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

REPO = Path(REPO_DIR)

print(f"▶  Pipeline started — {datetime.now():%Y-%m-%d %H:%M:%S}")
print("=" * 72)
print(f"  Target  : {DEAL['target_company']}")
print(f"  Deal    : {DEAL['deal_type'].title()}  |  ${DEAL['deal_value']:,}")
print(f"  Deal ID : {DEAL['deal_id']}")
print("=" * 72)
print("ADK will spawn 7 agents and make ~20 external API calls.")
print("MCP servers (SEC EDGAR, Legal DB) start automatically as subprocesses.")
print()

# Build command — prefer the installed 'adk' binary, fall back to python -m
_adk_bin = shutil.which("adk")
if _adk_bin:
    _cmd = [_adk_bin, "run", "app/", "--replay", "replay.json", "--in_memory"]
else:
    _cmd = [
        sys.executable, "-m", "google.adk",
        "run", "app/", "--replay", "replay.json", "--in_memory",
    ]

proc = subprocess.Popen(
    _cmd,
    cwd=REPO,
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,   # merge stderr so everything appears inline
    text=True,
    bufsize=1,                   # line-buffered for real-time streaming
    env={**os.environ},
)

# Pre-feed 'y\n' to auto-accept the save_report_locally confirmation prompt.
# ADK reads from stdin when require_confirmation=True is set on a tool.
try:
    proc.stdin.write("y\n" * 60)
    proc.stdin.flush()
    proc.stdin.close()
except OSError:
    pass  # process may have already exited

PIPELINE_OUTPUT = []
try:
    for _line in proc.stdout:
        print(_line, end="", flush=True)
        PIPELINE_OUTPUT.append(_line)
    proc.wait(timeout=720)          # 12-minute hard cap
except subprocess.TimeoutExpired:
    proc.kill()
    print("\n⚠  Timed out after 12 minutes. Partial results may still be in reports/.")
except KeyboardInterrupt:
    proc.terminate()
    print("\n⚠  Interrupted.")

PIPELINE_OUTPUT = "".join(PIPELINE_OUTPUT)
_rc = proc.returncode if proc.returncode is not None else -1
_icon = "✓" if _rc == 0 else "✗"
print(f"\n{_icon} Pipeline finished  (exit {_rc})  —  {datetime.now():%H:%M:%S}")

In [ ]:
# ── Step 6: Display results ───────────────────────────────────────────────────

from IPython.display import HTML, Image, display
from pathlib import Path

REPORTS = Path(REPO_DIR) / "reports" / DEAL["deal_id"]

# ── Risk matrix chart ─────────────────────────────────────────────────────────
chart = REPORTS / "risk_matrix.png"
if chart.exists():
    print("📊  Risk Matrix")
    display(Image(filename=str(chart), width=800))
else:
    print(f"ℹ  Risk matrix not found at {chart}")
    print("   The risk_assessor agent generates this — it runs after all 5 research")
    print("   agents complete. A 429 quota error mid-pipeline prevents it.")

# ── PDF report ────────────────────────────────────────────────────────────────
pdf = REPORTS / "due_diligence_report.pdf"
if pdf.exists():
    _kb = pdf.stat().st_size // 1024
    print(f"\n📄  PDF Report  ({_kb} KB)")
    # Kaggle renders relative paths in HTML links as download URLs
    display(HTML(
        f'<a href="{pdf}" download="{pdf.name}">'
        f'⬇&nbsp;Download {pdf.name}</a>'
    ))
else:
    print(f"\nℹ  PDF not found at {pdf}")
    print("   Requires the full pipeline to complete, including report_generator")
    print("   confirming the save_report_locally tool call.")

# ── All generated files ───────────────────────────────────────────────────────
if REPORTS.exists():
    files = sorted(REPORTS.iterdir())
    print(f"\nAll files in reports/{DEAL['deal_id']}/:")
    for f in files:
        print(f"  {f.name:<40}  {f.stat().st_size:>10,} bytes")
else:
    print(f"\nℹ  No reports directory found at {REPORTS}")
    print("   Run Step 5 first.")

## Notes and customisation

### Running a different company

Edit `DEAL` in **Step 3** and re-run from Step 3 onwards:

```python
DEAL = {
    "target_company"   : "Microsoft",        # any US public company
    "industry"         : "cloud computing",
    "deal_value"       : 200_000_000,
    "deal_type"        : "acquisition",
    "deal_id"          : "msft_demo_001",
    "deal_jurisdictions": ["US"],
    "scope"            : "Full due diligence",
    "acquirer_description": "Enterprise software acquirer",
}
```

Any company with public SEC filings works (Apple, Tesla, Palantir, etc.).
Private companies return `data_available: false` for EDGAR-based tools but
the pipeline completes gracefully.

### Downloading output files

After the run, go to **Kaggle → Data → Output** to download:
- `reports/{deal_id}/risk_matrix.png` — 5 × 5 risk heatmap
- `reports/{deal_id}/due_diligence_report.pdf` — full PDF

### MCP servers

Both stdio MCP servers (`sec_edgar_server`, `legal_db_server`) are spawned
**automatically** by ADK as subprocesses — no manual startup step is needed.

### Quota errors (429 RESOURCE_EXHAUSTED)

If you see `429` mid-pipeline:
1. Enable billing on your Google Cloud / AI Studio project (removes the free-tier cap)
2. Or wait ~24 h for the daily quota to reset and re-run

### Security plugins running on every agent turn

| Plugin | What it does |
|--------|--------------|
| `AuditPlugin` | Structured JSON audit log via structlog |
| `PIIRedactionPlugin` | Scrubs names / emails / phone numbers (presidio) |
| `FinancialGuardrailsPlugin` | Blocks financial figures without a public source URL |